# TP3 — 01: Carga y validacion del dataset

**Objetivo:** cargar los dos archivos del dataset Sentiment140, validar su estructura
y dejar constancia explicita de que el trabajo usa **todos los datos disponibles**
(requisito mandatorio de la consigna).

Archivos (ver `CONSIGNA.md`):

- `training.1600000.processed.noemoticon.csv`: 1.600.000 tweets etiquetados automaticamente.
- `testdata.manual.2009.06.14.csv`: 498 tweets etiquetados a mano.

Los CSV no traen encabezado; las columnas documentadas son
`target, id, date, query, user, text`.

In [1]:
import pandas as pd

from utils import (
    cargar_training,
    cargar_test,
    etiquetas,
    TARGET,
    TARGET_LABELS,
    TRAIN_CSV,
    TEST_CSV,
)

pd.set_option("display.max_colwidth", 120)

## Carga completa

Se cargan **todos** los registros de ambos archivos (`nrows=None`). La consigna admite
muestras unicamente para depurar codigo, nunca para los resultados finales.

In [2]:
train = cargar_training()  # nrows=None -> dataset completo, requisito mandatorio
test = cargar_test()

print(f"Training: {len(train):>9,} filas x {train.shape[1]} columnas  ({TRAIN_CSV.name})")
print(f"Test:     {len(test):>9,} filas x {test.shape[1]} columnas  ({TEST_CSV.name})")

Training: 1,600,000 filas x 6 columnas  (training.1600000.processed.noemoticon.csv)
Test:           498 filas x 6 columnas  (testdata.manual.2009.06.14.csv)


In [3]:
# Verificacion dura del requisito mandatorio: si algun dia el archivo se trunca
# o se reemplaza por una muestra, esta celda corta la ejecucion.
assert len(train) == 1_600_000, "El training NO esta completo"
assert len(test) == 498, "El test manual NO esta completo"
print("OK: el analisis usa el dataset COMPLETO (1.600.000 tweets de training y 498 de test).")

OK: el analisis usa el dataset COMPLETO (1.600.000 tweets de training y 498 de test).


In [4]:
train.head()

,target,id,date,query,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by texting it... and might cry as a result School today also. Blah!
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Managed to save 50% The rest go out of bounds
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all. i'm mad. why am i here? because I can't see you all over there."


## Distribucion de clases

En Sentiment140 la columna `target` codifica el sentimiento: `0` = negativo,
`2` = neutral, `4` = positivo.

In [5]:
def distribucion(df, nombre):
    d = df[TARGET].value_counts().sort_index().to_frame("filas")
    d["sentimiento"] = [TARGET_LABELS[t] for t in d.index]
    d["proporcion"] = (d["filas"] / len(df)).round(4)
    print(f"--- {nombre} ---")
    print(d[["sentimiento", "filas", "proporcion"]], "\n")

distribucion(train, "Training")
distribucion(test, "Test manual")

--- Training ---
       sentimiento   filas  proporcion
target                                
0         negativo  800000         0.5
4         positivo  800000         0.5 

--- Test manual ---
       sentimiento  filas  proporcion
target                               
0         negativo    177      0.3554
2          neutral    139      0.2791
4         positivo    182      0.3655 



**Hallazgo estructural clave** (condiciona todo el TP):

- El training esta **perfectamente balanceado**: 800.000 negativos y 800.000 positivos.
- El training **no tiene clase neutral (2)**; el test manual si la tiene.
- Por eso el modelo principal sera **binario** (negativo vs positivo) y la clase
  neutral se tratara como extension exploratoria en la notebook 05.

In [6]:
# El archivo viene ORDENADO por clase: primero todos los negativos, despues todos
# los positivos. Cualquier split debe hacerse con shuffle + estratificacion.
print("target de las primeras 3 filas:", train[TARGET].head(3).tolist())
print("target de las ultimas 3 filas: ", train[TARGET].tail(3).tolist())
print("posicion del primer positivo:  ", (train[TARGET] == 4).idxmax())

target de las primeras 3 filas: [0, 0, 0]
target de las ultimas 3 filas:  [4, 4, 4]
posicion del primer positivo:   800000


## Calidad de datos

Chequeos basicos: nulos, contenido de `query` y duplicados.

In [7]:
print("Nulos por columna:")
print(train.isna().sum())
print()
print("query en training:", train["query"].unique())
print("Cantidad de queries distintas en test:", test["query"].nunique())
print("Ejemplos de queries del test:", sorted(test["query"].unique())[:8])

Nulos por columna:
target    0
id        0
date      0
query     0
user      0
text      0
dtype: int64

query en training: <ArrowStringArray>
['NO_QUERY']
Length: 1, dtype: str
Cantidad de queries distintas en test: 81
Ejemplos de queries del test: ['"booz allen"', '"naive bayes"', '"night at the museum"', '"twitter api"', '40d', '50d', 'Bobby Flay', 'Danny Gokey']


In [8]:
ids_duplicados = train["id"].duplicated().sum()
textos_duplicados = train["text"].duplicated().sum()

# Textos que aparecen mas de una vez CON etiquetas distintas -> ruido de etiquetado.
etiquetas_por_texto = train.groupby("text")[TARGET].nunique()
textos_conflictivos = (etiquetas_por_texto > 1).sum()

print(f"IDs duplicados:                      {ids_duplicados:>6,}")
print(f"Textos duplicados:                   {textos_duplicados:>6,}")
print(f"Textos con etiquetas conflictivas:   {textos_conflictivos:>6,}")

IDs duplicados:                       1,685
Textos duplicados:                   18,534
Textos con etiquetas conflictivas:    2,225


In [9]:
# Ejemplos de textos repetidos con etiquetas opuestas (mismo tweet, distinto target).
conflictivos = etiquetas_por_texto[etiquetas_por_texto > 1].index[:5]
cols = [TARGET, "user", "text"]
ejemplo = train[train["text"].isin(conflictivos)].sort_values("text")[cols].head(10).copy()
ejemplo["sentimiento"] = etiquetas(ejemplo[TARGET])
ejemplo[["sentimiento", "user", "text"]]

,sentimiento,user,text
385331,negativo,FunStarLiz,"British weather is back i see! Oh well Birtney, london and ciaraaaa in 5 dayssss"
1394129,positivo,FunStarLiz,"British weather is back i see! Oh well Birtney, london and ciaraaaa in 5 dayssss"
507399,negativo,Aryy1,I love you
1272390,positivo,thatLenakid,I love you
184535,negativo,macpoulet67,"Raining tomorrow afternoon but its going to be very nice at 5 59! ! Hope it changes, I want it to be nicee in the a..."
1077756,positivo,macpoulet67,"Raining tomorrow afternoon but its going to be very nice at 5 59! ! Hope it changes, I want it to be nicee in the a..."
730048,negativo,Jessica_Tucker,That is all.
1030537,positivo,bradbury731,That is all.
118605,negativo,mckyliecooper,Uhm.. science! -.- Verrrry boring and LONG Grrr.
968532,positivo,mckyliecooper,Uhm.. science! -.- Verrrry boring and LONG Grrr.


## Decision sobre duplicados

Existen ~1.7 mil IDs duplicados, ~18.5 mil textos duplicados y ~2.2 mil textos que
aparecen con **etiquetas conflictivas** (el mismo texto como positivo y como negativo).
Esto es ruido del etiquetado automatico original (basado en emoticones).

**Decision:** se conservan todos los registros, por dos motivos:

1. La consigna exige usar el dataset completo.
2. El volumen de conflicto es marginal (~0,1% de las filas) y no altera el balance.

El costo es un **techo de performance**: ningun modelo puede clasificar bien un texto
que aparece con las dos etiquetas. Se retoma como limitacion en la conclusion.

## Resumen de la notebook

- Training completo: **1.600.000 tweets**, balanceado 50/50 negativo/positivo, sin nulos.
- Test manual completo: **498 tweets** con las tres clases (incluye neutral).
- El archivo de training esta ordenado por clase -> los splits usaran `shuffle=True` + `stratify`.
- Duplicados y conflictos de etiqueta documentados y conservados.